In [102]:
import torch
from datasets import load_dataset



import math
import regex as re
import tiktoken
ds = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    cache_dir="/Users/amannindra/hf_datasets"
)
print(ds)
import time

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import torch.nn as nn
from torch.nn import functional as F

import matplotlib.pyplot as plt


Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [2]:
ds.num_rows

2119719

In [3]:
print("DS Rows: ", ds.num_rows/100)

data_range = math.floor((ds.num_rows)/100)

# for i in range(data_range):
#     if "text" not in ds[i]:
#         print(ds[i])
#     if i % (math.floor(data_range/25)) == 0:
#         print(f"Row: {i}/{data_range}")

DS Rows:  21197.19


In [4]:
longest = 0
word = ""
tokens = []
infinity = math.inf

for i in range(data_range):
    if "text" not in ds[i]:
        print(ds[i])
    else: 
        # word = ds[i]["text"]
        # print(map(int, word.encode("utf-8")))
        # tokens.append(map(int, word.encode("utf-8")))
        
        
        if len(ds[i]["text"]) > longest:
            longest = len(ds[i]["text"])
            word = ds[i]["text"]   
        if len(ds[i]["text"]) < infinity:
            infinity = len(ds[i]["text"])
        

# print(longest)

# print(word.encode("utf-8"))

print(f"Shortest Word: {infinity}")
longest_tokens = list(map(int, word.encode("utf-8")))
longest_tokens


Shortest Word: 205


[84,
 111,
 109,
 32,
 97,
 110,
 100,
 32,
 76,
 105,
 108,
 121,
 32,
 119,
 101,
 114,
 101,
 32,
 116,
 119,
 105,
 110,
 115,
 32,
 119,
 104,
 111,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 112,
 108,
 97,
 121,
 32,
 119,
 105,
 116,
 104,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 84,
 104,
 101,
 121,
 32,
 104,
 97,
 100,
 32,
 109,
 97,
 110,
 121,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 32,
 111,
 102,
 32,
 100,
 105,
 102,
 102,
 101,
 114,
 101,
 110,
 116,
 32,
 99,
 111,
 108,
 111,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 115,
 104,
 97,
 112,
 101,
 115,
 46,
 32,
 84,
 104,
 101,
 121,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 98,
 117,
 105,
 108,
 100,
 32,
 116,
 111,
 119,
 101,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 104,
 111,
 117,
 115,
 101,
 115,
 32,
 97,
 110,
 100,
 32,
 99,
 97,
 114,
 115,
 32,
 119,
 105,
 116,
 104,
 32,
 116,
 104,
 101,
 105,
 114,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 83,
 111,
 1

In [5]:

def gets_stats(get_token):
    count = {}

    for i in range(len(get_token) - 1):
        if tuple(get_token[i:i + 2]) not in count.keys():
            count[tuple(get_token[i:i + 2])] = 1
        else:
            count[tuple(get_token[i:i + 2])] += 1
    return count
            
k = gets_stats(longest_tokens)      
gets_stats("Hello World")
# print(sorted(((v,k) for k,v in k.items()), reverse = True))



{('H', 'e'): 1,
 ('e', 'l'): 1,
 ('l', 'l'): 1,
 ('l', 'o'): 1,
 ('o', ' '): 1,
 (' ', 'W'): 1,
 ('W', 'o'): 1,
 ('o', 'r'): 1,
 ('r', 'l'): 1,
 ('l', 'd'): 1}

In [6]:
top_pairs = max(k, key=k.get)
top_pairs



(104, 101)

In [7]:
def merge(ids, pairs, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pairs[0] and ids[i + 1] == pairs[1]:
            newids.append(idx)
            i+=2
        else:
            newids.append(ids[i])
            i += 1
    return newids

        
tokens2 = merge(longest_tokens, top_pairs, 256) 
print(f"Original len: {len(longest_tokens)}, Merge: {len(tokens2)} ")
tokens2

Original len: 4540, Merge: 4320 


[84,
 111,
 109,
 32,
 97,
 110,
 100,
 32,
 76,
 105,
 108,
 121,
 32,
 119,
 101,
 114,
 101,
 32,
 116,
 119,
 105,
 110,
 115,
 32,
 119,
 104,
 111,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 112,
 108,
 97,
 121,
 32,
 119,
 105,
 116,
 104,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 84,
 256,
 121,
 32,
 104,
 97,
 100,
 32,
 109,
 97,
 110,
 121,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 32,
 111,
 102,
 32,
 100,
 105,
 102,
 102,
 101,
 114,
 101,
 110,
 116,
 32,
 99,
 111,
 108,
 111,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 115,
 104,
 97,
 112,
 101,
 115,
 46,
 32,
 84,
 256,
 121,
 32,
 108,
 105,
 107,
 101,
 100,
 32,
 116,
 111,
 32,
 98,
 117,
 105,
 108,
 100,
 32,
 116,
 111,
 119,
 101,
 114,
 115,
 32,
 97,
 110,
 100,
 32,
 104,
 111,
 117,
 115,
 101,
 115,
 32,
 97,
 110,
 100,
 32,
 99,
 97,
 114,
 115,
 32,
 119,
 105,
 116,
 104,
 32,
 116,
 256,
 105,
 114,
 32,
 98,
 108,
 111,
 99,
 107,
 115,
 46,
 32,
 83,
 111,
 109,
 101,
 116,
 1

In [8]:

# ids = list(tokens)
# print(ids)
# print(len(ids))


num_merge_2 = 20

merges_2 = {}



idx = 256
for j in range(math.floor(data_range/4)):
    word = ds[j]["text"]
    tokens = list(map(int, word.encode("utf-8")))
    ids = list(tokens)
    for i in range(num_merge_2):
        stats = gets_stats(ids)
        top_pairs = max(stats, key=stats.get)
        if top_pairs in merges_2:
            continue
        idx += 1
        ids = merge(ids, top_pairs, idx)
        merges_2[top_pairs] = idx
        
    if j % (math.floor(data_range/25)) == 0:
        print(f"Row: {j}/{math.floor(data_range/4)}")
        

vocab_size = 276
num_merge = vocab_size - 256
print("Part2")
idx = 256
merges = {}
ids = list(longest_tokens)
for i in range(num_merge):
    stats = gets_stats(ids)
    top_pairs = max(stats, key=stats.get)
    idx += 1
    ids = merge(ids, top_pairs, idx)
    merges[top_pairs] = idx




Row: 0/5299
Row: 847/5299
Row: 1694/5299
Row: 2541/5299
Row: 3388/5299
Row: 4235/5299
Row: 5082/5299
Part2


In [9]:
print(f"new merges: {len(merges)}, old merges: {len(merges_2)}")

print(merges)

print()

print(merges_2)


new merges: 20, old merges: 42
{(104, 101): 257, (32, 116): 258, (100, 32): 259, (258, 257): 260, (46, 32): 261, (121, 32): 262, (97, 110): 263, (258, 111): 264, (32, 98): 265, (101, 32): 266, (100, 260): 267, (116, 32): 268, (83, 257): 269, (115, 32): 270, (101, 114): 271, (104, 97): 272, (261, 269): 273, (263, 259): 274, (111, 107): 275, (257, 114): 276}

{(104, 101): 257, (32, 115): 258, (32, 116): 259, (101, 100): 260, (258, 104): 261, (97, 110): 262, (101, 32): 263, (32, 119): 264, (259, 257): 265, (257, 114): 266, (105, 116): 267, (32, 110): 268, (32, 102): 269, (261, 97): 270, (270, 114): 271, (262, 100): 272, (105, 108): 273, (44, 32): 274, (108, 263): 275, (105, 114): 276, (100, 32): 277, (116, 32): 278, (32, 97): 279, (32, 104): 280, (115, 32): 281, (121, 32): 282, (116, 104): 283, (110, 100): 284, (97, 284): 285, (114, 285): 286, (101, 114): 287, (46, 32): 288, (32, 98): 289, (32, 112): 290, (195, 162): 291, (291, 226): 292, (292, 130): 293, (293, 172): 294, (105, 110): 295,

In [10]:
def encode(text, merge_method):
    tokens = list(text.encode("utf-8"))
    while True:
        stats = gets_stats(tokens)
        # print(stats)
        pair = min(stats, key=lambda p: merge_method.get(p, float("inf")))
        if pair not in merge_method:
            # print("pair not in merges")
            break # nothing else can be paired
        idx = merge_method[pair]
        tokens = merge(tokens, pair, idx) 
    return tokens



print(encode("Hello World", merges))
print(encode("Hello World", merges_2))

[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100]
[72, 101, 297, 111, 32, 87, 298, 108, 100]


In [11]:
vocab = {idx: bytes([idx]) for idx in range(256)}
# print(vocab)

for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    
vocab_2 = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges_2.items():
    vocab_2[idx] = vocab_2[p0] + vocab_2[p1]


# print(vocab)

def decode(ids, vocab_method):
    tokens = b"".join(vocab_method[idx] for idx in ids)
    text = tokens.decode("utf-8", errors ="replace")
    return text

print(decode(ids, vocab))
print(decode(ids, vocab_2))


Tom and Lily were twins who liked to play with blocks. They had many blocks of different colors and shapes. They liked to build towers and houses and cars with their blocks. Sometimes they shared their blocks, and sometimes they fought over them.

One day, Tom wanted to build a big tower with all the blocks. He did not want to share with Lily. He took all the blocks and put them in a corner. Lily was sad and angry. She wanted to play with some blocks too. She asked Tom to give her some blocks, but Tom said no. He said they were all his blocks.

Lily waited for Tom to finish his tower. She hoped he would let her play with it when he was done. But Tom took a long time to complete his tower. He wanted to make it very tall and strong. He added more and more blocks to his tower. Lily got bored and lonely. She looked around for something else to play with.

She saw a book on the shelf. It was a book of news. It had pictures and words about what was happening in the world. Lily liked to look 

In [12]:
print(decode(encode("Hello World", merges), vocab))
valText = "I cannot express enough how grateful I am for your incredible tutorials. As a 44-year-old South Sudanese individual with three children and a full-time job, recently relocated from the UK to the US, my days are filled to the brim with responsibilities. However, I have made it a priority to utilize every ounce of my free time to follow your series of lectures. I must say, I have never been as excited about learning a new topic as I am now. Your clear explanations and engaging content have ignited a passion for learning within me that I never knew existed. Thank you from the bottom of my heart for all that you do."
valText2 = decode(encode(valText, merges), vocab)
print(valText == valText2)


print(decode(encode("Hello World", merges_2), vocab_2))
valText = "I cannot express enough how grateful I am for your incredible tutorials. As a 44-year-old South Sudanese individual with three children and a full-time job, recently relocated from the UK to the US, my days are filled to the brim with responsibilities. However, I have made it a priority to utilize every ounce of my free time to follow your series of lectures. I must say, I have never been as excited about learning a new topic as I am now. Your clear explanations and engaging content have ignited a passion for learning within me that I never knew existed. Thank you from the bottom of my heart for all that you do."
valText2 = decode(encode(valText, merges_2), vocab_2)
print(valText == valText2)

Hello World
True
Hello World
True


In [13]:
print(encode("Hello World", merges))
print(encode("Hello World", merges_2))

[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100]
[72, 101, 297, 111, 32, 87, 298, 108, 100]


In [14]:
decode([72, 101, 108, 108, 111, 32, 87, 111, 114,108, 100], vocab)
decode([72, 101, 297, 111, 32, 87, 298, 108, 100], vocab_2)

'Hello World'

In [15]:
count = 0 

divisble = 250
total_rows = math.floor(ds.num_rows/divisble)


start_time = time.perf_counter()


for i in range(total_rows):
    text = ds[i]["text"]
    if decode(encode(text, merges), vocab) == text:
        count += 1
    if i % total_rows == 0:
        print(f"Row {i}/{total_rows}")

print(f"Accuracy: {(count/total_rows)*100}%")


end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")



# 1. Record the starting time

count = 0

start_time = time.perf_counter()


for i in range(total_rows):
    text = ds[i]["text"]
    if decode(encode(text, merges_2), vocab_2) == text:
        count += 1
    if i % total_rows == 0:
        print(f"Row {i}/{total_rows}")

print(f"Accuracy: {(count/total_rows)*100}%")


end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")


Row 0/8478
Accuracy: 100.0%
Execution time: 23.404440 seconds
Row 0/8478
Accuracy: 100.0%
Execution time: 38.347689 seconds


In [ ]:
class LLM_Dataset(Dataset):
    def __init__(self, text_length = 200, encode = encode, decode = decode, merge_method = merges, vocab_method = vocab, device = torch.device("cpu")):
        self.device = device
        self.encode = encode
        self.decode = decode
        self.text_length = text_length
        self.merge = merge_method
        self.vocab = vocab_method
        
    def __getitem__(self, idx):
        text = ds[idx]["text"]  
        text_encode = self.encode(text, self.merge)
        if len(text_encode) > self.text_length:
            text_encode = text_encode[:self.text_length]
        else: 
            text_encode = text_encode + [0] * (self.text_length - len(text_encode))
            
        return torch.tensor(text_encode)
    def __len__(self):
        return ds.num_rows
    
    
    
LLM = LLM_Dataset(200, encode, decode, merges, vocab, torch.device("cpu"))

print(len(LLM[32]))
print(LLM[32])


200
tensor([ 79, 110,  99, 266, 117, 112, 111, 110,  32,  97, 258, 105, 109, 101,
         44, 260, 114, 266, 119,  97, 270,  97,  32, 108, 105, 116, 116, 108,
        101, 265, 111,  97, 116, 261,  84, 257, 265, 111,  97, 268, 119,  97,
        115, 265, 108, 117, 266, 274, 105, 268, 108, 105, 107, 101, 100, 264,
         32, 102, 108, 111,  97, 268, 111, 110, 260,  32, 119,  97, 116, 271,
        261,  79, 110, 266, 100,  97, 121,  44, 260,  32, 115, 117, 110,  32,
        119,  97, 270, 118, 271, 262, 104, 111, 268, 263, 267,  32, 119,  97,
        116, 271, 265, 101, 103, 263, 264,  32, 100, 114, 262, 117, 112, 261,
         84, 257, 265, 111,  97, 268, 119,  97, 270, 115,  97, 259,  98, 101,
         99,  97, 117, 115, 266, 105, 268,  99, 111, 117, 108, 259, 110, 111,
        268, 102, 108, 111,  97, 268, 263, 121, 109, 111, 114, 101,  46,  10,
         10,  65, 265, 105, 114, 259, 115,  97, 119, 260, 265, 111,  97, 268,
        274,  97, 115, 107, 101, 100,  44,  32,  34,  87, 10

In [41]:
train_divisible = 100

train_set = math.floor(math.floor(ds.num_rows/train_divisible)*0.8)
print(f"Trainin Set: {train_set}")


Trainin Set: 16957


In [89]:
batch_size = 4
block_size = len(merges) + 256
num_workers = 0
device = torch.device("cpu")


LLM_dataset = LLM_Dataset(block_size, encode, decode, merges, vocab, device)
train_dataloader = DataLoader(LLM_dataset, batch_size = batch_size, shuffle=True, num_workers = num_workers)




# batch_features, batch_labels = len(train_dataloader)

In [90]:
for batch in train_dataloader:
    x = batch
    break
for batch in train_dataloader:
    y = batch
    break

In [91]:
print(x.shape)
print(y.shape)

torch.Size([4, 276])
torch.Size([4, 276])


In [ ]:
torch.manual_seed(1337)


class BigramLangaugeModel(nn.Module):
    def __init__(self, vocab_input):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_input, vocab_input)
    def forward(self, idx, targets=None):
        
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else: 
            B, T, C = logits.shape
            print(f"B: {B}, T: {T}, C:{C} ")
            logits = logits.view(B*T, C)
            
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx
            

# print(block_size + 256)


# print(type(x), type(y))

m = BigramLangaugeModel(vocab_input=block_size + 256)
logits, loss = m(x,y)


idx = torch.zeros((1,1), dtype = torch.long)
print(f'generate output: {m.generate(idx, torch.tensor(1))[0].tolist()}')

print(f"vocab: {len(vocab)}")
print(f"vocab 2: {len(vocab_2)}")



r = decode(m.generate(idx, torch.tensor(100))[0].tolist(), vocab_2)
print(r)
# print(logits.shape) 

# print(loss.shape)


B: 4, T: 276, C:532 
generate output: [0, 363]
vocab: 276
vocab 2: 298


KeyError: 478